# Initial Condition Sensitivity — Inflow & Outflow (Part 1)

**Purpose:** Quantifies how sensitive the simulated flood hydrograph is to the
initial soil saturation state, for both the T = 500-year and T = 5000-year
design storms.

**What it does:**
- Loads TALSIM results for three initial condition (IC) scenarios:
  dry (21.58 %), medium (41.71 %), and saturated (100 %)
- Produces a 4-panel figure:
  - (a) Inflow  | T = 500yr  — IC dry / medium / saturated
  - (b) Inflow  | T = 5000yr — IC dry / medium / saturated
  - (c) Outflow | T = 500yr  — IC dry / medium / saturated
  - (d) Outflow | T = 5000yr — IC dry / medium / saturated

**User settings:** Edit only the USER SETTINGS block at the top  
**Input:** TALSIM `.WEL` output folders for each IC scenario  
**Output:** 4-panel IC sensitivity PNG

---

In [ ]:
FOLDER_IC1 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_21.58_60min")   # dry
FOLDER_IC2 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_41.71_60min\DVWK")   # medium
FOLDER_IC3 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_100_60min")  # saturated

In [ ]:
# =============================================================================
# FIGURE GROUP 4 — Initial Condition Sensitivity
# Answers: "How sensitive is the result to soil saturation?"
# 4-panel figure:
#   (a) Inflow  | T = 500yr  — 3 lines: IC dry / medium / saturated
#   (b) Inflow  | T = 5000yr — 3 lines: IC dry / medium / saturated
#   (c) Outflow | T = 500yr  — 3 lines: IC dry / medium / saturated
#   (d) Outflow | T = 5000yr — 3 lines: IC dry / medium / saturated
#
# HOW TO USE:
#   Edit ONLY the USER SETTINGS block below, then run.
# =============================================================================

# %% Imports
from pathlib import Path
import re
import unicodedata
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# =============================================================================
# *** USER SETTINGS — EDIT ONLY HERE ***
# =============================================================================

# --- Folders (one per initial condition) ------------------------------------
FOLDER_IC1 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_21.58_60min")   # dry
FOLDER_IC2 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_41.71_60min\DVWK")   # medium
FOLDER_IC3 = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_100_60min")  # saturated

# --- WEL file name inside each event subfolder ------------------------------
WEL_NAME = "Neuer_Teich_r.WEL"

# --- Columns ----------------------------------------------------------------
COL_INFLOW  = "T001_1ZU"   # inflow  column
COL_OUTFLOW = "S005_1ZU"   # outflow column

# --- Initial condition labels -----------------------------------------------
IC_LABEL_1 = "IC = 21.58 %  (dry)"
IC_LABEL_2 = "IC = 41.71 %  (medium)"
IC_LABEL_3 = "IC = 100 %    (saturated)"

# --- Duration to plot -------------------------------------------------------
# Script auto-detects the critical duration (highest inflow peak) per IC.
# No need to set anything here.

# --- Return periods ---------------------------------------------------------
T1 = 500
T2 = 5000

# --- Colors (dry=blue → medium=orange → saturated=red) ---------------------
C_IC1 = "#08306b"   # dry       — deep blue
C_IC2 = "#f57c00"   # medium    — orange
C_IC3 = "#e00000"   # saturated — red

# --- Line widths ------------------------------------------------------------
LW = 1.5

# --- Output folder ----------------------------------------------------------
OUT_DIR = Path(r"C:\Users\raah\Desktop\Project_ZR\figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Fixed settings
# =============================================================================

re_folder = re.compile(
    r"^(?P<num>\d{3})_(?P<dauer>\d+(?:[.,]\d+)?)h_(?P<yr>\d+)yr$",
    re.IGNORECASE
)
SKIP_DURATIONS = {0.25, 0.5}

# =============================================================================
# %% Helpers
# =============================================================================

def merge_split_underscore_cols(cols):
    fixed, i = [], 0
    while i < len(cols):
        if (i + 1 < len(cols)
                and cols[i + 1].startswith("_")
                and re.match(r"^[A-Za-z0-9]+$", cols[i])):
            fixed.append(cols[i] + cols[i + 1]); i += 2
        else:
            fixed.append(cols[i]); i += 1
    return fixed

_date_re = re.compile(r"^\d{2}\.\d{2}\.\d{4}$")
_time_re = re.compile(r"^\d{2}:\d{2}$")

def _find_wel(folder, wel_name):
    for name in [wel_name, wel_name.lower(), wel_name + ".txt"]:
        p = folder / name
        if p.exists():
            return p
    raise FileNotFoundError(f"WEL not found in {folder}")

def parse_wel(path, value_col):
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Datum_Zeit"):
            header_idx = i; break
    if header_idx is None:
        raise ValueError(f"Header not found in {path}")
    cols = merge_split_underscore_cols(lines[header_idx].split())
    if value_col not in cols:
        raise KeyError(
            f"Column '{value_col}' not in {path.name}\n"
            f"Available: {[c for c in cols if '1ZU' in c or '1AB' in c]}"
        )
    vpos            = cols.index(value_col)
    expected_tokens = len(cols) + 1
    start = header_idx + 1
    while start < len(lines):
        s = lines[start].strip()
        if not s or s.startswith("*") or s.lstrip().startswith("-"):
            start += 1; continue
        if lines[start].split() and _date_re.match(lines[start].split()[0]):
            break
        start += 1

    def find_dt(buf, at=0):
        for j in range(at, len(buf) - 1):
            if _date_re.match(buf[j]) and _time_re.match(buf[j + 1]):
                return j
        return None

    times, values, buf = [], [], []
    for line in lines[start:]:
        if not line.strip() or line.strip().startswith("*"):
            continue
        buf.extend(line.split())
        while True:
            s0 = find_dt(buf)
            if s0 is None:
                if len(buf) > 10 * expected_tokens: buf = buf[-expected_tokens:]
                break
            if s0 > 0: buf = buf[s0:]
            s1 = find_dt(buf, 2)
            end = s1 if (s1 and s1 < expected_tokens) else expected_tokens
            if len(buf) < end: break
            rec = buf[:end]; buf = buf[end:]
            if len(rec) != expected_tokens: continue
            try:
                ts = dt.datetime.strptime(rec[0] + " " + rec[1], "%d.%m.%Y %H:%M")
            except Exception:
                ts = None
            try:
                val = float(rec[2:][vpos - 1].replace(",", "."))
            except Exception:
                continue
            times.append(ts); values.append(val)

    q = np.asarray(values, dtype=float)
    if times and all(t is not None for t in times):
        t0  = times[0]
        t_h = np.array([(t - t0).total_seconds() / 3600 for t in times])
    else:
        t_h = np.arange(len(q), dtype=float)
    return {"q": q, "t_h": t_h}


def load_duration(root, wel_name, value_col, return_period, fixed_duration=None):
    """
    Load one duration for a given return period and column.
    If fixed_duration is set → use that duration.
    If None → auto-detect critical (highest peak).
    Returns: {"q": array, "t_h": array, "dauer_h": float, "qmax": float}
    """
    candidates = {}
    for sf in sorted(root.iterdir()):
        if not sf.is_dir(): continue
        m = re_folder.match(sf.name)
        if not m: continue
        if int(m.group("yr")) != return_period: continue
        dauer_h = float(m.group("dauer").replace(",", "."))
        if dauer_h in SKIP_DURATIONS:
            continue
        if fixed_duration is not None and abs(dauer_h - fixed_duration) > 0.01:
            continue
        try:
            wel    = _find_wel(sf, wel_name)
            parsed = parse_wel(wel, value_col)
            if len(parsed["q"]) > 0:
                candidates[dauer_h] = parsed
        except Exception as e:
            print(f"  [SKIP] {sf.name}: {e}")

    if not candidates:
        print(f"  [WARNING] No data for T={return_period}yr in {root.name}")
        return None

    # Pick critical or fixed
    if fixed_duration is not None:
        d = list(candidates.keys())[0]
    else:
        d = max(candidates, key=lambda x: float(np.nanmax(candidates[x]["q"])))

    return {
        "q":       candidates[d]["q"],
        "t_h":     candidates[d]["t_h"],
        "dauer_h": d,
        "qmax":    float(np.nanmax(candidates[d]["q"])),
    }


# =============================================================================
# %% Panel plotter
# =============================================================================

def plot_panel(ax, series_list, panel_title):
    """
    Plot 3 IC lines on one panel.
    - No legend inside panel (shared legend below figure)
    - Annotation box inside panel: critical duration + Qmax per IC
    series_list = list of (data_dict, color, label) tuples.
    Returns list of (line_handle, short_label) for shared legend.
    """
    qmax_global = 0.0
    handles = []

    for data, color, label in series_list:
        if data is None:
            continue
        q = data["q"]
        t = data["t_h"]
        qmax_global = max(qmax_global, data["qmax"])

        line, = ax.plot(t, q,
                        color=color, linestyle="-", linewidth=LW,
                        alpha=0.92, zorder=3)
        handles.append((line, label))

        # Peak marker
        idx = int(np.nanargmax(q))
        ax.plot(t[idx], q[idx],
                marker="^", markersize=5,
                color=color,
                markeredgewidth=0.5, markeredgecolor="white",
                zorder=10)


    # --- Annotation box: separate colored text lines, auto-sized box ---
    items    = [(c, d['dauer_h'], d['qmax']) for d, c, l in series_list if d is not None]
    line_h   = 0.068
    x_pos    = 0.97
    y_start  = 0.97

    # Draw background box first
    n_items  = len(items)
    box_h    = n_items * line_h + 0.04
    box_w    = 0.50
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch(
        (x_pos - box_w, y_start - box_h + 0.01),
        box_w + 0.005, box_h,
        transform=ax.transAxes,
        boxstyle='round,pad=0.02',
        facecolor='white', edgecolor='0.55',
        linewidth=0.8, alpha=0.95, zorder=20,
        clip_on=False
    ))
    # Draw each text line in its color on top
    for k, (color, dauer_h, qmax) in enumerate(items):
        y_pos = y_start - 0.016 - k * line_h
        ax.text(
            x_pos - 0.01, y_pos,
            f'd={dauer_h:g}h  Qmax={qmax:.2f}m3/s',
            transform=ax.transAxes,
            fontsize=6.5, ha='right', va='top',
            fontfamily='DejaVu Sans', color=color,
            fontweight='bold' if k == n_items - 1 else 'normal',
            zorder=21
        )


    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=qmax_global * 1.18)
    ax.xaxis.set_major_locator(MaxNLocator(8, integer=False))
    ax.tick_params(axis="both", length=4)
    ax.set_xlabel("Time [h] (from event start)", fontsize=8)
    ax.set_ylabel("Discharge [m³/s]", fontsize=8)
    ax.set_title(panel_title, pad=7, fontsize=9, loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return handles


# =============================================================================
# %% Main
# =============================================================================

plt.rcParams.update({
    "font.family":    "DejaVu Sans",
    "font.size":       8,
    "axes.titlesize":  9,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth":  0.7,
    "grid.linewidth":  0.45,
    "grid.alpha":      0.35,
    "grid.linestyle":  "--",
    "axes.grid":       True,
    "axes.axisbelow":  True,
    "figure.dpi":      150,
})

print("=" * 60)
print("Figure Group 4 — Initial Condition Sensitivity")
print("=" * 60)

# Each IC finds its own critical duration independently.
# The script reads all durations per IC and picks the one with the highest peak.

ICs = [
    (FOLDER_IC1, C_IC1, IC_LABEL_1),
    (FOLDER_IC2, C_IC2, IC_LABEL_2),
    (FOLDER_IC3, C_IC3, IC_LABEL_3),
]

print("\nLoading series — each IC selects its own critical duration...")
inflow_T1  = [(load_duration(f, WEL_NAME, COL_INFLOW,  T1, None), c, l) for f, c, l in ICs]
inflow_T2  = [(load_duration(f, WEL_NAME, COL_INFLOW,  T2, None), c, l) for f, c, l in ICs]
outflow_T1 = [(load_duration(f, WEL_NAME, COL_OUTFLOW, T1, None), c, l) for f, c, l in ICs]
outflow_T2 = [(load_duration(f, WEL_NAME, COL_OUTFLOW, T2, None), c, l) for f, c, l in ICs]

# Print summary of selected durations
print("\nCritical durations selected:")
for (data, _, label), rp in [(d, T1) for d in inflow_T1] + [(d, T2) for d in inflow_T2]:
    if data:
        print(f"  {label:<30}  T = {rp} yr  →  {data['dauer_h']:g} h  (Qmax = {data['qmax']:.2f} m³/s)")

# 4-panel figure
# (a) Inflow  T1  | (b) Outflow T1
# (c) Inflow  T2  | (d) Outflow T2
FIG_WIDTH  = 6.30
FIG_HEIGHT = 7.00
fig, axes  = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

print("\nPlotting panels...")

# Collect legend handles from first panel (same ICs across all panels)
handles = plot_panel(axes[0, 0], inflow_T1,  f"(a)  Inflow  |  T = {T1} yr")
plot_panel(axes[0, 1], outflow_T1, f"(b)  Outflow  |  T = {T1} yr")
plot_panel(axes[1, 0], inflow_T2,  f"(c)  Inflow  |  T = {T2} yr")
plot_panel(axes[1, 1], outflow_T2, f"(d)  Outflow  |  T = {T2} yr")

fig.tight_layout()
fig.subplots_adjust(hspace=0.42, wspace=0.32, bottom=0.12)

# --- Shared horizontal legend below all panels ---
# Format: Initial Condition:  (color) 21.58 %  |  (color) 41.71 %  |  (color) 100 %
legend_handles = [h for h, _ in handles]
legend_labels  = [
    lbl.split("=")[1].strip().split("(")[0].strip() if "=" in lbl else lbl
    for _, lbl in handles
]
# Add "Initial Condition:" as a title to the legend
fig.legend(
    legend_handles, legend_labels,
    loc="lower center",
    ncol=len(handles),
    fontsize=7,
    title="Initial Condition",
    title_fontsize=7,
    framealpha=0.92,
    edgecolor="0.7",
    borderpad=0.5,
    handlelength=2.0,
    columnspacing=2.0,
    bbox_to_anchor=(0.5, 0.01),
)

outpng = OUT_DIR / "Fig_Group4_IC_Sensitivity.png"
fig.savefig(outpng, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"\nSaved: {outpng}")
print("In Word: Insert > Pictures > set width = 16.0 cm")

# =============================================================================
# %% COLUMN HELPER — uncomment to check available columns
# =============================================================================
#
# for sf in sorted(FOLDER_IC1.iterdir()):
#     if sf.is_dir():
#         wel = sf / WEL_NAME
#         if wel.exists():
#             lines = wel.read_text(encoding="utf-8", errors="replace").splitlines()
#             for line in lines:
#                 if line.strip().startswith("Datum_Zeit"):
#                     cols = line.split()
#                     print("1ZU (inflow) :", [c for c in cols if "1ZU" in c])
#                     print("1AB (outflow):", [c for c in cols if "1AB" in c])
#                     break
#         break